# Top-5 Equal-Weight S&P 500 Strategy

**Rule:** On the first trading day of each month, buy the top-`TOP_N` S&P 500 constituents by market-cap weight (rows 0–4 of `sp500_constituents_YYYYMM.csv`) and allocate capital **equally** across them (1/N each). Hold for the entire month and rebalance at the start of the next month.

Historical prices are fetched from Yahoo Finance via `yfinance`.

**Benchmark:** SPY (S&P 500 ETF)

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd
import yfinance as yf

CONSTITUENTS_DIR = Path("sp500_constituents")
INITIAL_CAPITAL   = 10_000   # USD
BENCHMARK         = "SPY"
TOP_N             = 5        # equal-weight across top-N symbols

## 1 · Build the monthly holdings table

One row per CSV file → the top-`TOP_N` stocks for that month (equal-weighted).

In [2]:
records = []
for csv_file in sorted(CONSTITUENTS_DIR.glob("sp500_constituents_*.csv")):
    month_tag = csv_file.stem.split("_")[-1]   # e.g. '202603'
    if len(month_tag) != 6:
        continue
    df = pd.read_csv(csv_file, usecols=["symbol"])
    top_symbols = df["symbol"].iloc[:TOP_N].tolist()   # rows 0…TOP_N-1
    records.append({
        "month"  : pd.to_datetime(month_tag, format="%Y%m"),
        "symbols": top_symbols,
    })

holdings = pd.DataFrame(records).sort_values("month").reset_index(drop=True)
print(f"Months loaded : {len(holdings)}")
print(f"Date range    : {holdings['month'].iloc[0]:%Y-%m} → {holdings['month'].iloc[-1]:%Y-%m}")
display(holdings.head(25))

Months loaded : 315
Date range    : 2000-01 → 2026-03


,month,symbols
0,2000-01-01,"[AIG, C, MSFT, CSCO, GE]"
1,2000-02-01,"[AIG, C, MSFT, CSCO, INTC]"
2,2000-03-01,"[AIG, C, MSFT, CSCO, INTC]"
3,2000-04-01,"[AIG, C, MSFT, CSCO, INTC]"
4,2000-05-01,"[AIG, C, CSCO, INTC, MSFT]"
5,2000-06-01,"[AIG, C, INTC, CSCO, MSFT]"
6,2000-07-01,"[AIG, C, INTC, MSFT, CSCO]"
7,2000-08-01,"[AIG, C, INTC, CSCO, MSFT]"
8,2000-09-01,"[AIG, C, INTC, CSCO, GE]"
9,2000-10-01,"[AIG, C, GE, CSCO, MSFT]"


## 2 · Download price data

In [ ]:
all_symbols = sorted(set(sym for syms in holdings["symbols"] for sym in syms) | {BENCHMARK})

start_date = holdings["month"].min().strftime("%Y-%m-01")
# fetch one extra month so the last holding has a full period
end_date   = (holdings["month"].max() + pd.DateOffset(months=2)).strftime("%Y-%m-01")

print(f"Downloading {all_symbols} from {start_date} to {end_date} …")
raw = yf.download(all_symbols, start=start_date, end=end_date,
                  auto_adjust=True, progress=False)

# keep only closing prices; flatten multi-index if multiple tickers
if isinstance(raw.columns, pd.MultiIndex):
    prices = raw["Close"].copy()
else:
    prices = raw[["Close"]].rename(columns={"Close": all_symbols[0]}).copy()

prices.index = pd.to_datetime(prices.index)
prices.sort_index(inplace=True)
print(f"Price matrix shape: {prices.shape}")
prices.tail(3)

## 3 · Backtest

In [ ]:
def monthly_return(series: pd.Series, entry: pd.Timestamp, exit_: pd.Timestamp) -> float:
    """Month-end to month-end return.

    Buy price  = last available close strictly before entry  (last close of previous month).
    Sell price = last available close on or before exit_     (last close of current month).
    """
    buy_series  = series.loc[: entry - pd.Timedelta(days=1)].dropna()
    sell_series = series.loc[entry : exit_].dropna()
    if buy_series.empty or sell_series.empty:
        return 0.0
    return sell_series.iloc[-1] / buy_series.iloc[-1] - 1


results = []
strat_value = INITIAL_CAPITAL
bench_value = INITIAL_CAPITAL

for i, row in holdings.iterrows():
    entry = row["month"]                                              # first calendar day of month
    exit_ = (
        holdings.loc[i + 1, "month"] - pd.Timedelta(days=1)          # day before next rebalance
        if i + 1 < len(holdings)
        else entry + pd.offsets.MonthEnd(1)                           # end of last month
    )

    syms = row["symbols"]
    # keep only symbols present in the price matrix
    tradeable = [s for s in syms if s in prices.columns]
    if not tradeable:
        print(f"  [WARN] no price data for any of {syms} — skipping {entry:%Y-%m}")
        continue

    # equal-weight: average return across available symbols
    weight     = 1.0 / len(tradeable)
    strat_ret  = sum(weight * monthly_return(prices[s], entry, exit_) for s in tradeable)
    bench_ret  = monthly_return(prices[BENCHMARK], entry, exit_)

    fund_start   = strat_value
    strat_value *= 1 + strat_ret
    bench_value *= 1 + bench_ret

    results.append({
        "month"       : entry,
        "symbols"     : ", ".join(tradeable),
        "n_held"      : len(tradeable),
        "fund_start"  : round(fund_start, 2),
        "strat_return": round(strat_ret * 100, 2),
        "bench_return": round(bench_ret * 100, 2),
        "strat_value" : round(strat_value, 2),
        "bench_value" : round(bench_value, 2),
    })

perf = pd.DataFrame(results)
print(f"Months backtested: {len(perf)}")
display(perf.tail(12))

## 4 · Monthly trade log

In [ ]:
if not perf.empty:
    log = perf[["month", "symbols", "n_held", "fund_start", "strat_return", "bench_return", "strat_value"]].copy()

    log["month"]        = log["month"].dt.strftime("%Y-%m")
    log["fund_start"]   = log["fund_start"].map("${:,.2f}".format)
    log["strat_value"]  = log["strat_value"].map("${:,.2f}".format)
    log["strat_return"] = log["strat_return"].map("{:+.2f}%".format)
    log["bench_return"] = log["bench_return"].map("{:+.2f}%".format)

    log.columns = ["Month", "Symbols held (equal-weight)", "# held", "Capital entering month", "Strategy return", "SPY return", "Portfolio value"]

    display(
        log.style
           .set_properties(**{"text-align": "right"})
           .set_properties(subset=["Month", "Symbols held (equal-weight)"], **{"text-align": "left"})
           .set_table_styles([{"selector": "th", "props": [("text-align", "center")]}])
    )

## 4 · Summary statistics

In [ ]:
if not perf.empty:
    total_strat = (perf["strat_value"].iloc[-1] / INITIAL_CAPITAL - 1) * 100
    total_bench = (perf["bench_value"].iloc[-1] / INITIAL_CAPITAL - 1) * 100
    win_months  = (perf["strat_return"] > perf["bench_return"]).sum()

    print(f"{'Metric':<32} {'Strategy':>12} {'SPY Benchmark':>15}")
    print("-" * 62)
    print(f"{'Initial capital':<32} {'$'+f'{INITIAL_CAPITAL:,.0f}':>12} {'$'+f'{INITIAL_CAPITAL:,.0f}':>15}")
    print(f"{'Final value':<32} {'$'+f'{perf.strat_value.iloc[-1]:,.2f}':>12} {'$'+f'{perf.bench_value.iloc[-1]:,.2f}':>15}")
    print(f"{'Total return (%)':<32} {total_strat:>11.2f}% {total_bench:>14.2f}%")
    print(f"{'Avg monthly return (%)':<32} {perf.strat_return.mean():>11.2f}% {perf.bench_return.mean():>14.2f}%")
    print(f"{'Best month (%)':<32} {perf.strat_return.max():>11.2f}% {perf.bench_return.max():>14.2f}%")
    print(f"{'Worst month (%)':<32} {perf.strat_return.min():>11.2f}% {perf.bench_return.min():>14.2f}%")
    print(f"{'Months outperforming SPY':<32} {win_months:>12} / {len(perf)}")
    print(f"{'Months tracked':<32} {len(perf):>12}")

## 5 · Portfolio value chart

In [ ]:
if not perf.empty:
    fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True,
                             gridspec_kw={"height_ratios": [3, 1]})

    # ── top panel: portfolio value ──────────────────────────────────────
    ax = axes[0]
    dates  = [perf["month"].iloc[0] - pd.DateOffset(months=1)] + perf["month"].tolist()
    s_vals = [INITIAL_CAPITAL] + perf["strat_value"].tolist()
    b_vals = [INITIAL_CAPITAL] + perf["bench_value"].tolist()

    ax.plot(dates, s_vals, linewidth=2, label=f"Top-{TOP_N} Equal-Weight", color="steelblue")
    ax.plot(dates, b_vals, linewidth=2, label="SPY Benchmark",             color="darkorange", linestyle="--")
    ax.axhline(INITIAL_CAPITAL, color="grey", linewidth=0.8, linestyle=":")
    ax.yaxis.set_major_formatter(mticker.StrMethodFormatter("${x:,.0f}"))
    ax.set_ylabel("Portfolio Value (USD)")
    ax.set_title(f"Top-{TOP_N} Equal-Weight S&P 500 Strategy  vs  SPY  (monthly rebalance)",
                 fontsize=12, fontweight="bold")
    ax.legend()
    ax.grid(axis="y", alpha=0.4)

    # annotate rank-1 symbol when the basket changes (avoids clutter)
    prev_syms = None
    for _, r in perf.iterrows():
        if r["symbols"] != prev_syms:
            label = r["symbols"].split(",")[0].strip()
            ax.annotate(label, xy=(r["month"], r["strat_value"]),
                        xytext=(0, 8), textcoords="offset points",
                        ha="center", fontsize=7, color="steelblue",
                        arrowprops=dict(arrowstyle="-", color="steelblue", lw=0.5))
            prev_syms = r["symbols"]

    # ── bottom panel: monthly return bars ───────────────────────────────
    ax2 = axes[1]
    width = 12
    ax2.bar([d - pd.Timedelta(days=7) for d in perf["month"]], perf["strat_return"],
            width=width, label="Strategy", color="steelblue", alpha=0.8)
    ax2.bar([d + pd.Timedelta(days=7) for d in perf["month"]], perf["bench_return"],
            width=width, label="SPY",      color="darkorange", alpha=0.8)
    ax2.axhline(0, color="black", linewidth=0.7)
    ax2.yaxis.set_major_formatter(mticker.StrMethodFormatter("{x:.0f}%"))
    ax2.set_ylabel("Monthly Return")
    ax2.legend(fontsize=8)
    ax2.grid(axis="y", alpha=0.4)

    plt.tight_layout()
    plt.savefig("sp500_constituents/top5_strategy.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Chart saved → sp500_constituents/top5_strategy.png")